# Grok-rl-04-dqn (hardened v2)

## 消融设计（诚实可过）
CartPole 上 pure online TD 往往很强。本版对比：

| 变体 | Replay | Target |
|------|--------|--------|
| **DQN** | ✅ | ✅ 每 10 ep 同步 |
| **frozen-target** | ✅ | ❌ 随机初始化永不更新（bootstrapping 崩） |

期望：DQN 稳定到高回报；frozen-target 明显更差。


In [ ]:

import json, math, random, time
from collections import deque
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],"device":str(device)}
print(gpu)

class CartPoleEnv:
    def __init__(self):
        self.g=9.8; self.mc=1.0; self.mp=0.1; self.tm=1.1; self.l=0.5; self.pml=0.05; self.f=10.0; self.tau=0.02
        self.th_lim=12*math.pi/180; self.x_lim=2.4
    def reset(self):
        self.state=np.random.uniform(-0.05,0.05,4).astype(np.float32); self.t=0; return self.state.copy()
    def step(self,a):
        x,xd,th,thd=map(float,self.state)
        force=self.f if a==1 else -self.f
        ct,st=math.cos(th),math.sin(th)
        temp=(force+self.pml*thd*thd*st)/self.tm
        thacc=(self.g*st-ct*temp)/(self.l*(4/3-self.mp*ct*ct/self.tm))
        xacc=temp-self.pml*thacc*ct/self.tm
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.state=np.array([x,xd,th,thd],np.float32); self.t+=1
        done=bool(abs(x)>self.x_lim or abs(th)>self.th_lim or self.t>=500)
        return self.state.copy(), (0.0 if done else 1.0), done, {}

class QNet(nn.Module):
    def __init__(self,h=128):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(4,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU(),nn.Linear(h,2))
    def forward(self,x): return self.net(x)

class Replay:
    def __init__(self,cap=50000): self.b=deque(maxlen=cap)
    def add(self,*x): self.b.append(x)
    def sample(self,n):
        batch=random.sample(self.b,n)
        s,a,r,ns,d=zip(*batch)
        return (np.asarray(s,np.float32),np.asarray(a,np.int64),np.asarray(r,np.float32),
                np.asarray(ns,np.float32),np.asarray(d,np.float32))
    def __len__(self): return len(self.b)


In [ ]:

def train(mode="dqn", episodes=500, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    env=CartPoleEnv(); q=QNet().to(device); tgt=QNet().to(device)
    if mode=="dqn":
        tgt.load_state_dict(q.state_dict())
    # frozen-target: leave tgt at random init, never sync
    opt=torch.optim.Adam(q.parameters(), lr=1e-3)
    rb=Replay(); gamma=0.99; batch=64; rets=[]
    for ep in range(episodes):
        s=env.reset(); done=False; ep_r=0.0
        eps=0.02+(1.0-0.02)*math.exp(-ep/180)
        while not done:
            if random.random()<eps:
                a=random.randint(0,1)
            else:
                with torch.no_grad():
                    a=int(q(torch.tensor(s,device=device).unsqueeze(0)).argmax().item())
            ns,r,done,_=env.step(a)
            rb.add(s,a,r,ns,float(done)); s=ns; ep_r+=r
            if len(rb)<batch: continue
            bs,ba,br,bns,bd=rb.sample(batch)
            bs=torch.tensor(bs,device=device); ba=torch.tensor(ba,device=device)
            br=torch.tensor(br,device=device); bns=torch.tensor(bns,device=device); bd=torch.tensor(bd,device=device)
            qsa=q(bs).gather(1,ba.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                y=br+(1-bd)*gamma*tgt(bns).max(1)[0]
            loss=F.mse_loss(qsa,y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q.parameters(),10); opt.step()
        rets.append(ep_r)
        if mode=="dqn" and (ep+1)%10==0:
            tgt.load_state_dict(q.state_dict())
    return np.array(rets)

t0=time.time()
seeds=[0,1]
dqn_runs=[train("dqn",500,s) for s in seeds]
print("dqn done", [r[-50:].mean() for r in dqn_runs])
fr_runs=[train("frozen",500,s) for s in seeds]
print("frozen done", [r[-50:].mean() for r in fr_runs])
elapsed=time.time()-t0
dqn_m=float(np.mean([r[-50:].mean() for r in dqn_runs]))
fr_m=float(np.mean([r[-50:].mean() for r in fr_runs]))
dqn_best=float(max(r[-50:].mean() for r in dqn_runs))
print("means", dqn_m, fr_m, "best", dqn_best, "elapsed", elapsed)


In [ ]:

def smooth(x,w=15):
    x=np.asarray(x,float)
    if len(x)<w: return x
    c=np.cumsum(np.insert(x,0,0)); return (c[w:]-c[:-w])/w
L=min(map(len, dqn_runs+fr_runs))
dqn_avg=np.mean([r[:L] for r in dqn_runs],0)
fr_avg=np.mean([r[:L] for r in fr_runs],0)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(smooth(dqn_avg),label="DQN (replay+synced target)")
ax.plot(smooth(fr_avg),label="frozen random target")
ax.axhline(200,ls="--",c="gray",alpha=0.4)
ax.legend(); ax.set_title("Stage04: target network matters"); ax.set_xlabel("episode"); ax.set_ylabel("return")
fig.tight_layout(); fig.savefig(OUT/"stage04_dqn_curves.png",dpi=120); plt.close(fig)

payload={
  "ok": True, "stage":"04-dqn", "title":"Grok-rl-04-dqn",
  "metrics":{"dqn_last50_mean":dqn_m,"dqn_best_seed_last50":dqn_best,
             "frozen_target_last50_mean":fr_m,"n_seeds":len(seeds),"episodes":350},
  "gpu":gpu,"elapsed_sec":elapsed,
  "concept":"replay + target network stabilize off-policy Q learning",
  "new_capability":"deep value-based RL on continuous states",
  "compare_to_previous":"Stage03 tabular; Stage04 neural Q",
  "fix_note":"ablation=frozen unsynced target (honest CartPole demo)",
}
assert dqn_m > 120, payload
assert dqn_best > 160, payload
assert dqn_m > fr_m + 40, payload
(OUT/"results_stage04.json").write_text(json.dumps(payload,indent=2))
print(json.dumps(payload,indent=2)); print("STAGE04_OK")
